# 7. Local Keras and Lite model wrappers

[All tutorials](https://github.com/bateni1380/landpy/tree/master/tutorials) · [Open in Colab](https://colab.research.google.com/github/bateni1380/landpy/blob/master/tutorials/7-model-wrappers.ipynb)

## Goal

Run both model wrappers using a tiny known function. This verifies the wrappers without depending on a pretrained model or external image.

## Setup

Run every cell in order. Python 3.10+ is required; use Python 3.10–3.13 for TensorFlow examples. The first cell installs the pinned release if needed. A CPU is enough.

In [1]:
import importlib.metadata
import subprocess
import sys
try:
    installed = importlib.metadata.version('landpy')
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != '0.1.0':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'landpy==0.1.0'])
import numpy as np
import matplotlib.pyplot as plt
from landpy import ImageLand, MarksLand, SquareMarks, QuadMarks, DualMarks, HeatLand, Filter, Debugger
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.titlesize': 12})

The model examples need the optional TensorFlow dependencies. Model weights download only when a model is constructed. Reuse the cache on later runs.

In [2]:
import importlib.util
if importlib.util.find_spec('tensorflow') is None or importlib.util.find_spec('tf_keras') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'landpy[tensorflow]==0.1.0'])
import warnings
warnings.filterwarnings("ignore", message=r"(?s).*tf.lite.Interpreter is deprecated.*", category=UserWarning)

## Steps

### 1. Save a small Keras model

Model is the abstract download/load contract. TFModel loads .keras and legacy .h5 files. A local `model_path` avoids downloading. Only load model files you trust.

In [3]:
import tensorflow as tf
tf.get_logger().setLevel("ERROR")
from pathlib import Path
import tempfile
from landpy.tf_model import TFModel
from landpy.tf_lite_model import TFLiteModel
workspace = Path(tempfile.mkdtemp(prefix='landpy-tutorial-'))
inputs = tf.keras.Input(shape=(2,))
outputs = tf.keras.layers.Dense(1, use_bias=False, kernel_initializer=tf.keras.initializers.Constant([[2.], [3.]]))(inputs)
keras_model = tf.keras.Model(inputs, outputs)
keras_path = workspace/'weighted_sum.keras'
keras_model.save(keras_path)
data = np.array([[4., 5.]], dtype=np.float32)
keras_wrapper = TFModel(model_path=keras_path, input_type=np.ndarray)
keras_result = keras_wrapper.inference(data)
print('Keras result:', keras_result)

Keras result: [[23.]]


### 2. Convert and run Lite

TFLiteModel checks the declared tensor shape, casts its dtype, and supports quantized inputs/outputs. A single output returns an array; multiple outputs return a list. The wrapper is single-input and fixed-shape.

In [4]:
converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
lite_path = workspace/'weighted_sum.tflite'
import contextlib
import io
with contextlib.redirect_stdout(io.StringIO()):
    lite_bytes = converter.convert()
lite_path.write_bytes(lite_bytes)
lite_wrapper = TFLiteModel(model_path=lite_path, input_type=np.ndarray)
lite_result = lite_wrapper.inference(data)
print('Lite result:', lite_result)

Lite result: [[23.]]


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


### 3. Verify a local model fingerprint

Remote HTTPS weights are downloaded atomically into a URL-specific cache, avoiding filename collisions. Built-in models pin checksums; a custom model can pass its own SHA-256. Failed downloads do not become valid cache entries.

In [5]:
import hashlib
fingerprint = hashlib.sha256(lite_path.read_bytes()).hexdigest()
verified = TFLiteModel(model_path=lite_path, sha256=fingerprint)
print('SHA-256:', fingerprint)
help(TFLiteModel.inference)

SHA-256: 3bd4b0b5f2aa30e48af24fd2685dcd5770867d607675567b6389685a24c22e5c
Help on function inference in module landpy.tf_lite_model:

inference(self, input)
    Run one input tensor using its declared shape and dtype.



## Checks

In [6]:
expected = np.array([[23.]], dtype=np.float32)
np.testing.assert_allclose(keras_result, expected)
np.testing.assert_allclose(lite_result, expected)
np.testing.assert_allclose(verified.inference(data), expected)
print('Both backends computed 2 × 4 + 3 × 5 = 23.')

Both backends computed 2 × 4 + 3 × 5 = 23.


## Next steps

Continue with tutorial 8 in the [tutorial index](https://github.com/bateni1380/landpy/tree/master/tutorials).